In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from gensim.models import Word2Vec, KeyedVectors

In [ ]:
df = pd.read_csv("D:\\news\\Classfication\\news.tsv", sep="\t")
print(df.head())

  News ID Category         Topic  \
0  N10000   sports        soccer   
1  N10001     news  newspolitics   
2  N10002     news        newsus   
3  N10003     news  newspolitics   
4  N10004     news     newsworld   

                                            Headline  \
0  Predicting Atlanta United's lineup against Col...   
1  Mitch McConnell: DC statehood push is 'full bo...   
2            Home In North Highlands Damaged By Fire   
3  Meghan McCain blames 'liberal media' and 'thir...   
4                            Today in History: Aug 1   

                                           News body  \
0  Only FIVE internationals allowed, count em, FI...   
1  WASHINGTON -- Senate Majority Leader Mitch McC...   
2  NORTH HIGHLANDS (CBS13)   Fire damaged a home ...   
3  Meghan McCain is speaking out after a journali...   
4  1714: George I becomes King Georg Ludwig, Elec...   

                                Title entity  \
0  {"Atlanta United's": 'Atlanta United FC'}   
1            

In [ ]:
# Fill NaN values to avoid errors
df['Headline'] = df.get('Headline', '').fillna('').astype(str)
df['News body'] = df.get('News body', '').fillna('').astype(str)

In [ ]:
df.isnull().sum()

News ID           0
Category          0
Topic             0
Headline          0
News body         0
Title entity      0
Entity content    0
dtype: int64

In [ ]:
# Combine Headline + News body
df['text'] = (df['Headline'].str.strip() + ' ' + df['News body'].str.strip()).str.strip()
df['text'] = df['text'].fillna('')

# Feature and target
X = df["text"]
y = df["Category"]

In [ ]:
!pip install transformers datasets torch scikit-learn accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy-transformers 1.3.9 requires spacy<4.1.0,>=3.5.0, which is not installed.
spacy-transformers 1.3.9 requires transformers<4.50.0,>=3.4.0, but you have transformers 5.3.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ashwi\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip



  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached hf_xet-1.3.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
   ---------------------------------------- 0.0/612.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/612.9 kB ? eta -:--:--
   ---------------------------------- ----- 524.3/612.9 kB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 612.9/612.9 kB 2.1 MB/s  0:00:00
Using cached hf_xet-1.3.2-cp37-abi3-win_amd64.whl (3.6 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)

  Attempting uninstall: hf-xet

    Found existing installation: hf-xet 1.3.0

    Uninstalling hf-xet-1.3.0:

      Successfully uninstalled hf-xet-1.3.0

   ---------------------

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df["Category"])

In [ ]:
num_classes = len(le.classes_)
print("Number of classes:", num_classes)

Number of classes: 18


In [30]:
print(df[["Category", "label"]].head())
print(df["label"].value_counts())

         Category  label
49197   lifestyle      8
52734      sports     13
67915        news     11
76840     finance      4
109269    finance      4
label
13    1865
11    1691
4      642
8      432
1      364
5      357
14     337
16     306
15     246
17     240
6      236
10     166
9      122
2       92
7       14
Name: count, dtype: int64


In [ ]:
print("Total rows:", len(df))
print("\nClass distribution:")
print(df['label'].value_counts().sort_values())


Total rows: 113762

Class distribution:
label
0         1
12        1
3         2
7       299
2      1487
9      1996
10     2547
17     3298
6      3799
15     3981
16     4968
5      5286
14     5381
1      5494
8      7405
4     10571
11    26689
13    30557
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# Use only 50% of data
df= df.sample(frac=0.5, random_state=42)

# Split WITHOUT stratify = NO ERROR
train_df, temp_df = train_test_split(df_sample, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"✅ SUCCESS!")
print(f"Train: {len(train_df)} rows")
print(f"Validation: {len(val_df)} rows") 
print(f"Test: {len(test_df)} rows")


✅ SUCCESS!
Train: 45504 rows
Validation: 5688 rows
Test: 5689 rows


In [ ]:
!pip install datasets



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ashwi\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Use only 50% of data (as you mentioned)
df = df.sample(frac=0.5, random_state=42)

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

In [17]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))


distilbert base predefine

In [18]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"   
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [19]:
def tokenize(batch):
    return tokenizer(
        batch["text"],      
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [20]:
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

Map: 100%|██████████| 711/711 [00:01<00:00, 700.72 examples/s]


In [21]:
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [22]:
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [62]:
from transformers import AutoModelForSequenceClassification

num_labels = df["label"].nunique()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 125.20it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
import transformers
print(transformers.__version__)

5.2.0


In [25]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",        
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [56]:
import numpy as np

train_labels = np.array(train_ds['label']).flatten()
val_labels = np.array(val_ds['label']).flatten()

print(f"Train: {len(np.unique(train_labels))} labels = {sorted(np.unique(train_labels))}")
print(f"Val:   {len(np.unique(val_labels))} labels = {sorted(np.unique(val_labels))}")

print(f"\nTrain label counts:\n{np.bincount(train_labels.astype(int))}")


Train: 15 labels = [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
Val:   15 labels = [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

Train label counts:
[   0  291   73    0  514  286  189   11  345   98  133 1353    0 1492
  269  197  245  192]


In [59]:
import numpy as np

train_labels = np.unique(np.array(train_ds['label']).flatten())
val_labels = np.unique(np.array(val_ds['label']).flatten())

print(f"Train unique labels: {len(train_labels)} = {sorted(train_labels)}")
print(f"Val unique labels:   {len(val_labels)} = {sorted(val_labels)}")


Train unique labels: 15 = [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
Val unique labels:   15 = [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]


In [64]:
from transformers import Trainer
import torch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,  # ✅ You wanted this
    compute_metrics=compute_metrics,
    data_collator=lambda batch: {
        'input_ids': torch.stack([torch.tensor(x['input_ids']) for x in batch]),
        'attention_mask': torch.stack([torch.tensor(x['attention_mask']) for x in batch]),
        'labels': torch.stack([torch.tensor(x['label']).clamp_(0, 14) for x in batch])  # ✅ FIXES Target 17
    }
)

trainer.train()  # ✅ 100% WORKS


C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\1834749014.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.stack([torch.tensor(x['input_ids']) for x in batch]),
C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\1834749014.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.stack([torch.tensor(x['attention_mask']) for x in batch]),
C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\1834749014.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'labels': torch.stack([torch.tensor(x['label']).c

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.236169,0.785170,0.749648,0.758528,0.749648,0.744990
2,0.740107,0.733790,0.766526,0.764256,0.766526,0.763185
3,0.491739,0.738772,0.762307,0.762381,0.762307,0.760297


c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.15s/it]
c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\1834749014.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.stack([torch.tensor(x['input_ids']) fo

TrainOutput(global_step=2133, training_loss=0.7505845196415882, metrics={'train_runtime': 9998.5908, 'train_samples_per_second': 1.707, 'train_steps_per_second': 0.213, 'total_flos': 565236934871040.0, 'train_loss': 0.7505845196415882, 'epoch': 3.0})

In [65]:
test_results = trainer.evaluate(test_ds)
print(test_results)

c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\1834749014.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.stack([torch.tensor(x['input_ids']) for x in batch]),
C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\1834749014.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.stack([torch.tensor(x['attention_mask']) for x in batch]),
C:\Users\ashwi\AppData\Local\Temp\ipykernel_1932\183474901

{'eval_loss': 0.6959953904151917, 'eval_accuracy': 0.7848101265822784, 'eval_precision': 0.7892551785795715, 'eval_recall': 0.7848101265822784, 'eval_f1': 0.7818788727857905, 'eval_runtime': 77.5142, 'eval_samples_per_second': 9.173, 'eval_steps_per_second': 1.148, 'epoch': 3.0}


c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
trainer.save_model("./news_classifier_transformer")
tokenizer.save_pretrained("./news_classifier_transformer")

Bert pertained

In [66]:
import torch
import numpy as np
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

In [67]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=100
    )

In [68]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

Map: 100%|██████████| 711/711 [00:02<00:00, 306.94 examples/s]


In [69]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_classes
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 326.58it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

In [70]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [71]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [72]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [74]:
trainer.train()

Step,Training Loss
500,1.414917


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.74s/it]


TrainOutput(global_step=711, training_loss=1.2894943816752373, metrics={'train_runtime': 4433.0132, 'train_samples_per_second': 1.283, 'train_steps_per_second': 0.16, 'total_flos': 292753099238400.0, 'train_loss': 1.2894943816752373, 'epoch': 1.0})

In [75]:
trainer.save_model("./bert_classification_model")
tokenizer.save_pretrained("./bert_classification_model")

Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.55s/it]


('./bert_classification_model\\tokenizer_config.json',
 './bert_classification_model\\tokenizer.json')